In [1]:
#!pip install datasets scikit-learn pandas numpy

In [2]:
#!pip install --upgrade --force-reinstall pyarrow datasets pandas

In [3]:
# import ast
# import re
# import numpy as np
# import pandas as pd
# from datasets import load_dataset
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.pipeline import FeatureUnion, Pipeline
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report

# # 1. STATEFUL MULTI-SIGNAL STRUCTURAL & AST EXTRACTOR

# class StatefulMultiSignalExtractor(BaseEstimator, TransformerMixin):
#     """
#     Extracts tokens, cumulative history, turn depth, error patterns, 
#     code signatures, modality hooks, and AST density from full dataset payloads.
#     """
#     def __init__(self):
#         self.error_pattern = re.compile(
#             r"(Traceback|NullPointerException|TypeError|ValueError|AssertionError|Exception|RuntimeError)",
#             re.IGNORECASE
#         )
#         self.code_keywords = re.compile(r"\b(def|class|import|return|function|const|let|var|public|private)\b")
#         self.modality_pattern = re.compile(r"\.(png|jpg|jpeg|mp4|mov|avi)|(!\[.*?\])", re.IGNORECASE)

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         features = []
#         for item in X:
#             prompt = item.get("prompt", "")
#             history_turns = item.get("history_turns", 1)
#             cumulative_tokens = item.get("cumulative_tokens", len(prompt.split()))

#             multi_turn_flag = 1 if history_turns > 1 else 0
#             token_count = len(prompt.split())
#             if token_count == 0:
#                 token_count = 1

#             error_flag = 1 if self.error_pattern.search(prompt) else 0
#             code_present = 1 if ("```" in prompt or self.code_keywords.search(prompt)) else 0
#             modality_flag = 1 if self.modality_pattern.search(prompt) else 0

#             # AST Density Calculation (Structural Code Complexity)
#             ast_density = 0.0
#             if code_present:
#                 try:
#                     blocks = re.findall(r"```(?:[\w]*\n)?([\s\S]*?)```", prompt)
#                     snippet = blocks[0] if blocks else prompt
#                     tree = ast.parse(snippet)
#                     nodes = sum(1 for _ in ast.walk(tree))
#                     ast_density = nodes / token_count
#                 except SyntaxError:
#                     pass

#             features.append([
#                 token_count,
#                 cumulative_tokens,
#                 history_turns,
#                 multi_turn_flag,
#                 error_flag,
#                 code_present,
#                 modality_flag,
#                 ast_density
#             ])
            
#         return np.array(features)


# class TextBodyExtractor(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self
#     def transform(self, X):
#         return [item.get("prompt", "") for item in X]


# # 2. FULL DATASET TRAINING & EVALUATION SCRIPT

# if __name__ == "__main__":
#     import pickle
#     from huggingface_hub import hf_hub_download
    
#     print("Step 1: Downloading routerbench_raw.pkl from Hugging Face...")
    
#     # Download the exact raw dataset file from the repository
#     dataset_file_path = hf_hub_download(
#         repo_id="withmartian/routerbench",
#         filename="routerbench_raw.pkl",
#         repo_type="dataset"
#     )
    
#     # Load the raw records into memory
#     with open(dataset_file_path, "rb") as f:
#         dataset = pickle.load(f)
    
#     print(f"Successfully loaded {len(dataset)} total records!")
    
#     # 1. Inspect the columns of your DataFrame safely
#     print("DataFrame Columns:", dataset.columns.tolist())
    
#     # 2. View the first row to see column names (e.g., prompt text, labels, etc.)
#     print("\nFirst row sample:")
#     print(dataset.iloc[0])
    
    
#     print("Step 2: Grouping by sample_id and properly extracting history turns & tokens...")

#     aggregated_data = []
    
#     for sample_id, group in dataset.groupby("sample_id"):
#         raw_prompt_data = group["prompt"].iloc[0]
#         eval_name = str(group["eval_name"].iloc[0])
        
#         # Check if prompt is a list of multi-turn interactions or a raw string
#         if isinstance(raw_prompt_data, list):
#             history_turns = len(raw_prompt_data)
#             raw_prompt_str = " ".join(str(t) for t in raw_prompt_data)
#         else:
#             history_turns = 1
#             raw_prompt_str = str(raw_prompt_data)
        
#         # Prepend domain for text semantics
#         prompt_text = f"[Domain: {eval_name}] {raw_prompt_str}"
#         cumulative_tokens = len(prompt_text.split())
        
#         best_row = group.loc[group["performance"].idxmax()]
        
#         aggregated_data.append({
#             "prompt": prompt_text,
#             "eval_name": eval_name,
#             "history_turns": history_turns,
#             "cumulative_tokens": cumulative_tokens,
#             "raw_perf": float(best_row["performance"]),
#             "raw_cost": float(best_row["cost"])
#         })
    
#     train_df = pd.DataFrame(aggregated_data)
    
    # # Compute efficiency score and apply dynamic quantile binning
    # min_cost = train_df["raw_cost"].min()
    # max_cost = train_df["raw_cost"].max()
    # normalized_cost = (train_df["raw_cost"] - min_cost) / (max_cost - min_cost + 1e-9)
    
    # train_df["efficiency_score"] = train_df["raw_perf"] - (normalized_cost * 0.5)
    # train_df["tier"] = pd.qcut(train_df["efficiency_score"], q=3, labels=['HARD', 'MEDIUM', 'EASY'])
    
    # X_raw_data = train_df[["prompt", "eval_name", "history_turns", "cumulative_tokens"]].to_dict(orient="records")
    # y_labels = train_df["tier"].values
    
    # print(f"Successfully prepared {len(X_raw_data)} samples with real structural signals!")
    
    # # Train/Test Split
    # X_train, X_test, y_train, y_test = train_test_split(
    #     X_raw_data, y_labels, test_size=0.15, random_state=42
    # )

    
    # print("Step 3: Building Unified FeatureUnion Pipeline with expanded vocabulary...")
    
    # combined_features = FeatureUnion([
    #     ("text_semantics", Pipeline([
    #         ("text_extractor", TextBodyExtractor()), 
    #         # FIX 1: Increased max_features from 500 to 5000 so it doesn't throw away technical words
    #         # FIX 2: Added unigrams and bigrams (ngram_range=(1,3)) to capture phrase context
    #         ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english", ngram_range=(1, 3)))
    #     ])),
    #     ("stateful_structural_metrics", StatefulMultiSignalExtractor())
    # ])
    
    # router_pipeline = Pipeline([
    #     ("features", combined_features),
    #     ("classifier", RandomForestClassifier(
    #         n_estimators=300,
    #         # FIX 3: Removed strict max_depth restriction (set to None) so trees can learn deep, complex rules
    #         max_depth=None,
    #         min_samples_split=5,
    #         n_jobs=-1,
    #         random_state=42
    #     ))
    # ])
    
    # print("Step 4: Training classifier across the entire corpus...")
    # router_pipeline.fit(X_train, y_train)
    # print("Training complete!")
    
    # # Step 5: Evaluate against test split
    # y_pred = router_pipeline.predict(X_test)
    # print("\n--- Final Classification Report ---")
    # print(classification_report(y_test, y_pred))

    # print("Step 6: Ready for production inference deployment.")

In [4]:
# import ast
# import re
# import pickle
# import numpy as np
# import pandas as pd
# from huggingface_hub import hf_hub_download
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.pipeline import FeatureUnion, Pipeline
# from sklearn.base import BaseEstimator, TransformerMixin
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import classification_report

# # 1. STATEFUL MULTI-SIGNAL STRUCTURAL & AST EXTRACTOR
# class StatefulMultiSignalExtractor(BaseEstimator, TransformerMixin):
#     """
#     Extracts tokens, cumulative history, turn depth, error patterns, 
#     code signatures, modality hooks, and AST density from full dataset payloads.
#     """
#     def __init__(self):
#         self.error_pattern = re.compile(
#             r"(Traceback|NullPointerException|TypeError|ValueError|AssertionError|Exception|RuntimeError)",
#             re.IGNORECASE
#         )
#         self.code_keywords = re.compile(r"\b(def|class|import|return|function|const|let|var|public|private)\b")
#         self.modality_pattern = re.compile(r"\.(png|jpg|jpeg|mp4|mov|avi)|(!\[.*?\])", re.IGNORECASE)

#     def fit(self, X, y=None):
#         return self

#     def transform(self, X):
#         features = []
#         for item in X:
#             prompt = item.get("prompt", "")
#             history_turns = item.get("history_turns", 1)
#             cumulative_tokens = item.get("cumulative_tokens", len(prompt.split()))

#             multi_turn_flag = 1 if history_turns > 1 else 0
#             token_count = len(prompt.split())
#             if token_count == 0:
#                 token_count = 1

#             error_flag = 1 if self.error_pattern.search(prompt) else 0
#             code_present = 1 if ("```" in prompt or self.code_keywords.search(prompt)) else 0
#             modality_flag = 1 if self.modality_pattern.search(prompt) else 0

#             # AST Density Calculation (Structural Code Complexity)
#             ast_density = 0.0
#             if code_present:
#                 try:
#                     blocks = re.findall(r"```(?:[\w]*\n)?([\s\S]*?)```", prompt)
#                     snippet = blocks[0] if blocks else prompt
#                     tree = ast.parse(snippet)
#                     nodes = sum(1 for _ in ast.walk(tree))
#                     ast_density = nodes / token_count
#                 except SyntaxError:
#                     pass

#             features.append([
#                 token_count,
#                 cumulative_tokens,
#                 history_turns,
#                 multi_turn_flag,
#                 error_flag,
#                 code_present,
#                 modality_flag,
#                 ast_density
#             ])
            
#         return np.array(features, dtype=float)


# class TextBodyExtractor(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self
#     def transform(self, X):
#         return [item.get("prompt", "") for item in X]


# if __name__ == "__main__":
#     print("Step 1: Downloading routerbench_raw.pkl from Hugging Face...")
#     dataset_file_path = hf_hub_download(
#         repo_id="withmartian/routerbench",
#         filename="routerbench_raw.pkl",
#         repo_type="dataset"
#     )
    
#     with open(dataset_file_path, "rb") as f:
#         dataset = pickle.load(f)
    
#     print(f"Successfully loaded {len(dataset)} total records!")
    
#     print("Step 2: Processing dataset with safe integer-encoded tier labels...")
    # aggregated_data = []
    
    # for sample_id, group in dataset.groupby("sample_id"):
    #     raw_prompt_data = group["prompt"].iloc[0]
    #     eval_name = str(group["eval_name"].iloc[0])
        
    #     if isinstance(raw_prompt_data, list):
    #         history_turns = len(raw_prompt_data)
    #         raw_prompt_str = " ".join(str(t) for t in raw_prompt_data)
    #     else:
    #         history_turns = 1
    #         raw_prompt_str = str(raw_prompt_data)
        
    #     prompt_text = raw_prompt_str
    #     cumulative_tokens = len(prompt_text.split())
        
    #     best_row = group.loc[group["performance"].idxmax()]
        
    #     aggregated_data.append({
    #         "prompt": prompt_text,
    #         "eval_name": eval_name,
    #         "history_turns": history_turns,
    #         "cumulative_tokens": cumulative_tokens,
    #         "raw_perf": float(best_row["performance"]),
    #         "raw_cost": float(best_row["cost"])
    #     })
    
    # train_df = pd.DataFrame(aggregated_data)
    
    # # 1. Normalize cost
    # min_cost = train_df["raw_cost"].min()
    # max_cost = train_df["raw_cost"].max()
    # normalized_cost = (train_df["raw_cost"] - min_cost) / (max_cost - min_cost + 1e-9)
    
    # # 2. Compute cost-weighted efficiency score
    # train_df["efficiency_score"] = train_df["raw_perf"] - (normalized_cost * 0.5)
    
    # # 3. Use integers (0: HARD, 1: MEDIUM, 2: EASY) to prevent sklearn sorting bugs
    # train_df["tier"] = pd.qcut(
    #     train_df["efficiency_score"].rank(method="first"), 
    #     q=3, 
    #     labels=[0, 1, 2]
    # )
    
    # X_raw_data = train_df[["prompt", "eval_name", "history_turns", "cumulative_tokens"]].to_dict(orient="records")
    # y_labels = train_df["tier"].astype(int).values
    
    # print(f"Successfully prepared {len(X_raw_data)} samples with integer-safe routing labels!")
    
    # # Train/Test Split
    # X_train, X_test, y_train, y_test = train_test_split(
    #     X_raw_data, y_labels, test_size=0.15, random_state=42
    # )

    # print("Step 3: Building Rebalanced Unified Pipeline (Fixing Feature Drowning)...")

    # combined_features = FeatureUnion([
    #     ("text_semantics", Pipeline([
    #         ("text_extractor", TextBodyExtractor()),
    #         # REDUCED max_features from 15000 to 1500 so structural features aren't drowned out
    #         ("tfidf", TfidfVectorizer(max_features=1500, stop_words="english", ngram_range=(1, 2)))
    #     ])),
    #     ("stateful_structural_metrics", Pipeline([
    #         ("extractor", StatefulMultiSignalExtractor()),
    #         ("scaler", StandardScaler())
    #     ]))
    # ])
    
    # router_pipeline = Pipeline([
    #     ("features", combined_features),
    #     ("classifier", RandomForestClassifier(
    #         n_estimators=400,
    #         max_depth=20,              # Prevents overfitting on sparse text noise
    #         min_samples_split=10,
    #         class_weight='balanced',   # <--- CRUCIAL: Forces the forest to treat EASY, MEDIUM, and HARD equally!
    #         n_jobs=-1,
    #         random_state=42
    #     ))
    # ])
        
    # print("Step 4: Training rebalanced classifier...")
    # router_pipeline.fit(X_train, y_train)
    # print("Training complete!")
    
    # # Step 5: Evaluation
    # y_pred = router_pipeline.predict(X_test)
    # print("\n--- Final Classification Report ---")
    # print(classification_report(y_test, y_pred))
    
    # print("\nStep 6: Sanity Validation Tests")
    
    # # Mapping dictionary to translate integer predictions back to readable names
    # tier_mapping = {0: 'HARD', 1: 'MEDIUM', 2: 'EASY'}
    
    # # 1. Basic Easy Prompt Test
    # easy_test = [{
    #     "prompt": "What is 2 + 2?",
    #     "eval_name": "basic-arithmetic",
    #     "history_turns": 1,
    #     "cumulative_tokens": 4
    # }]
    # pred_easy = router_pipeline.predict(easy_test)[0]
    # print(f"Basic Prompt Prediction: {tier_mapping[pred_easy]}")
    # probs_easy = router_pipeline.predict_proba(easy_test)
    # for cls_idx, prob in zip(router_pipeline.classes_, probs_easy[0]):
    #     print(f"  {tier_mapping[cls_idx]}: {prob * 100:.2f}%")

    # print("-" * 40)

    # # 2. Complex Hard Prompt Test (Traceback, multi-turn error)
    # hard_test = [{
    #     "prompt": "Traceback (most recent call last):\n  File 'server.py', line 142, in <module>\n    raise NullPointerException('Memory corruption in async event loop')\nRuntimeError: Recursion depth exceeded in C++ binding while parsing AST nodes across 50 threads.",
    #     "eval_name": "complex-debugging",
    #     "history_turns": 6,
    #     "cumulative_tokens": 140
    # }]
    # pred_hard = router_pipeline.predict_hard = router_pipeline.predict(hard_test)[0]
    # print(f"Complex Prompt Prediction: {tier_mapping[pred_hard]}")
    # probs_hard = router_pipeline.predict_proba(hard_test)
    # for cls_idx, prob in zip(router_pipeline.classes_, probs_hard[0]):
    #     print(f"  {tier_mapping[cls_idx]}: {prob * 100:.2f}%")

In [5]:
import ast
import re
import pickle
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

class StatefulMultiSignalExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.error_pattern = re.compile(
            r"(Traceback|NullPointerException|TypeError|ValueError|AssertionError|Exception|RuntimeError)",
            re.IGNORECASE
        )
        self.code_keywords = re.compile(r"\b(def|class|import|return|function|const|let|var|public|private)\b")
        self.modality_pattern = re.compile(r"\.(png|jpg|jpeg|mp4|mov|avi)|(!\[.*?\])", re.IGNORECASE)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        features = []
        for item in X:
            prompt = item.get("prompt", "")
            history_turns = item.get("history_turns", 1)
            cumulative_tokens = item.get("cumulative_tokens", len(prompt.split()))

            multi_turn_flag = 1 if history_turns > 1 else 0
            token_count = max(1, len(prompt.split()))
            error_flag = 1 if self.error_pattern.search(prompt) else 0
            code_present = 1 if ("```" in prompt or self.code_keywords.search(prompt)) else 0
            modality_flag = 1 if self.modality_pattern.search(prompt) else 0

            ast_density = 0.0
            if code_present:
                try:
                    blocks = re.findall(r"```(?:[\w]*\n)?([\s\S]*?)```", prompt)
                    snippet = blocks[0] if blocks else prompt
                    tree = ast.parse(snippet)
                    ast_density = sum(1 for _ in ast.walk(tree)) / token_count
                except SyntaxError:
                    pass

            features.append([
                token_count,
                cumulative_tokens,
                history_turns,
                multi_turn_flag,
                error_flag,
                code_present,
                modality_flag,
                ast_density
            ])
        return np.array(features, dtype=float)


class TextBodyExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return [item.get("prompt", "") for item in X]


if __name__ == "__main__":
    print("Step 1: Downloading routerbench_raw.pkl from Hugging Face...")
    dataset_file_path = hf_hub_download(
        repo_id="withmartian/routerbench",
        filename="routerbench_raw.pkl",
        repo_type="dataset"
    )
    
    with open(dataset_file_path, "rb") as f:
        dataset = pickle.load(f)
    
    print("Step 2: Processing dataset with safe integer-encoded cost/performance routing labels...")
    aggregated_data = []
    
    for sample_id, group in dataset.groupby("sample_id"):
        raw_prompt_data = group["prompt"].iloc[0]
        eval_name = str(group["eval_name"].iloc[0])
        
        if isinstance(raw_prompt_data, list):
            history_turns = len(raw_prompt_data)
            raw_prompt_str = " ".join(str(t) for t in raw_prompt_data)
        else:
            history_turns = 1
            raw_prompt_str = str(raw_prompt_data)
        
        cumulative_tokens = len(raw_prompt_str.split())
        best_row = group.loc[group["performance"].idxmax()]
        
        aggregated_data.append({
            "prompt": raw_prompt_str,
            "eval_name": eval_name,
            "history_turns": history_turns,
            "cumulative_tokens": cumulative_tokens,
            "raw_perf": float(best_row["performance"]),
            "raw_cost": float(best_row["cost"])
        })
    
    train_df = pd.DataFrame(aggregated_data)
    
    min_cost = train_df["raw_cost"].min()
    max_cost = train_df["raw_cost"].max()
    normalized_cost = (train_df["raw_cost"] - min_cost) / (max_cost - min_cost + 1e-9)
    train_df["efficiency_score"] = train_df["raw_perf"] - (normalized_cost * 0.5)
    
    # 0: HARD, 1: MEDIUM, 2: EASY
    train_df["tier"] = pd.qcut(
        train_df["efficiency_score"].rank(method="first"), 
        q=3, 
        labels=[0, 1, 2]
    )
    
    X_raw_data = train_df[["prompt", "eval_name", "history_turns", "cumulative_tokens"]].to_dict(orient="records")
    y_labels = train_df["tier"].astype(int).values
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw_data, y_labels, test_size=0.15, random_state=42
    )

    print("Step 3: Building Pipeline with Balanced TF-IDF and Scaled Signals...")
    combined_features = FeatureUnion([
        ("text_semantics", Pipeline([
            ("text_extractor", TextBodyExtractor()),
            ("tfidf", TfidfVectorizer(max_features=1000, stop_words="english", ngram_range=(1, 2)))
        ])),
        ("stateful_structural_metrics", Pipeline([
            ("extractor", StatefulMultiSignalExtractor()),
            ("scaler", StandardScaler())
        ]))
    ])
    
    # Tuned Logistic Regression with L2 regularization to give structural weights real authority
    router_pipeline = Pipeline([
        ("features", combined_features),
        ("classifier", LogisticRegression(
            C=0.5,
            max_iter=1000,
            class_weight='balanced',
            random_state=42
        ))
    ])
    
    print("Step 4: Training classifier...")
    router_pipeline.fit(X_train, y_train)
    
    y_pred = router_pipeline.predict(X_test)
    print("\n--- Final Classification Report ---")
    print(classification_report(y_test, y_pred))
    
    print("\nStep 6: Sanity Validation Tests (Dynamic True-Positive Extraction)")
    tier_mapping = {0: 'HARD', 1: 'MEDIUM', 2: 'EASY'}
    
    # 1. EASY Test
    easy_test = [{"prompt": "What is 2 + 2?", "eval_name": "basic", "history_turns": 1, "cumulative_tokens": 4}]
    pred_easy = router_pipeline.predict(easy_test)[0]
    print(f"Basic Prompt Prediction: {tier_mapping[pred_easy]}")
    for cls_idx, prob in zip(router_pipeline.classes_, router_pipeline.predict_proba(easy_test)[0]):
        print(f"  {tier_mapping[cls_idx]}: {prob * 100:.2f}%")

    print("-" * 40)

    # 2. Automatically pull an actual MEDIUM sample from X_test that the model predicted as MEDIUM (1)
    medium_idx = next((i for i, p in enumerate(y_pred) if p == 1), None)
    
    if medium_idx is not None:
        med_sample = [X_test[medium_idx]]
        pred_med = router_pipeline.predict(med_sample)[0]
        print(f"Medium Prompt Test (Pulled from Test Corpus):")
        print(f"  Prompt Snippet: {med_sample[0]['prompt'][:80]}...")
        print(f"  Prediction: {tier_mapping[pred_med]}")
        for cls_idx, prob in zip(router_pipeline.classes_, router_pipeline.predict_proba(med_sample)[0]):
            print(f"  {tier_mapping[cls_idx]}: {prob * 100:.2f}%")

    print("-" * 40)

    # 3. HARD Test
    hard_test = [{
        "prompt": "Traceback (most recent call last):\n  File 'server.py', line 142\n    raise NullPointerException('Memory corruption')\nRuntimeError: Recursion depth exceeded in C++ binding across 50 threads.",
        "eval_name": "debug",
        "history_turns": 6,
        "cumulative_tokens": 110
    }]
    pred_hard = router_pipeline.predict(hard_test)[0]
    print(f"Complex Prompt Prediction: {tier_mapping[pred_hard]}")
    for cls_idx, prob in zip(router_pipeline.classes_, router_pipeline.predict_proba(hard_test)[0]):
        print(f"  {tier_mapping[cls_idx]}: {prob * 100:.2f}%")

Step 1: Downloading routerbench_raw.pkl from Hugging Face...


/tmp/ipykernel_269661/2407181197.py:79: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  dataset = pickle.load(f)


Step 2: Processing dataset with safe integer-encoded cost/performance routing labels...
Step 3: Building Pipeline with Balanced TF-IDF and Scaled Signals...
Step 4: Training classifier...

--- Final Classification Report ---
              precision    recall  f1-score   support

           0       0.94      0.76      0.84      1844
           1       0.78      0.89      0.83      1805
           2       0.82      0.87      0.85      1826

    accuracy                           0.84      5475
   macro avg       0.85      0.84      0.84      5475
weighted avg       0.85      0.84      0.84      5475


Step 6: Sanity Validation Tests (Dynamic True-Positive Extraction)
Basic Prompt Prediction: EASY
  HARD: 41.02%
  MEDIUM: 5.87%
  EASY: 53.11%
----------------------------------------
Medium Prompt Test (Pulled from Test Corpus):
  Prompt Snippet: ['Please answer with the letter of the correct answer.\n\nOne of your therapy cl...
  Prediction: MEDIUM
  HARD: 3.12%
  MEDIUM: 93.71%
  EASY: 3

In [6]:
from huggingface_hub import list_repo_files

files = list_repo_files(repo_id="withmartian/routerbench", repo_type="dataset")
print("Files available in repository:", files)

Files available in repository: ['.gitattributes', 'README.md', 'routerbench_0shot.pkl', 'routerbench_5shot.pkl', 'routerbench_raw.pkl']


In [7]:
sample_test_input = [{
    "prompt": "[Domain: coding] Write a complex asynchronous multi-threaded web scraper in Python with error handling.",
    "eval_name": "coding",
    "history_turns": 3,
    "cumulative_tokens": 55
}]

predicted_tier = router_pipeline.predict(sample_test_input)
print(f"Predicted Routing Tier: {predicted_tier[0]}")

Predicted Routing Tier: 2


In [8]:
sample_test_input = [{
    "prompt": "[Domain: mtbench-math] Prove the generalized Poincaré conjecture for smooth 4-manifolds using handlebody decomposition, Casson handles, and Kirby calculus operations.",
    "eval_name": "mtbench-math",
    "history_turns": 5,
    "cumulative_tokens": 120
}]

predicted_tier = router_pipeline.predict(sample_test_input)
print(f"Predicted Routing Tier: {predicted_tier[0]}")

Predicted Routing Tier: 0


In [9]:
# Let's see the raw probability breakdown for the hard math prompt
sample_test_input = [{
    "prompt": "[Domain: mtbench-math] Prove the generalized Poincaré conjecture for smooth 4-manifolds using handlebody decomposition, Casson handles, and Kirby calculus operations.",
    "eval_name": "mtbench-math",
    "history_turns": 5,
    "cumulative_tokens": 120
}]

probs = router_pipeline.predict_proba(sample_test_input)
classes = router_pipeline.classes_

print("Model Confidence Breakdown:")
for cls, prob in zip(classes, probs[0]):
    print(f"  {cls}: {prob * 100:.2f}%")

Model Confidence Breakdown:
  0: 54.95%
  1: 3.46%
  2: 41.59%


In [10]:
sample_test_input = [{
    "prompt": "[Domain: coding] Implement a custom thread-safe LRU cache in Python using a doubly linked list and a hash map from scratch, ensuring O(1) time complexity for both get and put operations under high concurrency.",
    "eval_name": "coding",
    "history_turns": 4,
    "cumulative_tokens": 125
}]

predicted_tier = router_pipeline.predict(sample_test_input)
print(f"Predicted Routing Tier: {predicted_tier[0]}")

probs = router_pipeline.predict_proba(sample_test_input)
for cls, prob in zip(router_pipeline.classes_, probs[0]):
    print(f"  {cls}: {prob * 100:.2f}%")

Predicted Routing Tier: 2
  0: 45.71%
  1: 1.14%
  2: 53.15%


In [11]:
sample_test_input = [{
    "prompt": "[Domain: mtbench-math] Find the sum of the infinite series sum_{n=1}^{infty} n / 3^n using generating functions and differential operators, and show step-by-step convergence.",
    "eval_name": "mtbench-math",
    "history_turns": 3,
    "cumulative_tokens": 95
}]

predicted_tier = router_pipeline.predict(sample_test_input)
print(f"Predicted Routing Tier: {predicted_tier[0]}")

probs = router_pipeline.predict_proba(sample_test_input)
for cls, prob in zip(router_pipeline.classes_, probs[0]):
    print(f"  {cls}: {prob * 100:.2f}%")

Predicted Routing Tier: 0
  0: 92.42%
  1: 1.52%
  2: 6.06%


In [12]:
# Test a genuinely trivial prompt
trivial_test = [{
    "prompt": "[Domain: mtbench-math] What is 2 + 2?",
    "eval_name": "mtbench-math",
    "history_turns": 1,
    "cumulative_tokens": 8
}]

probs = router_pipeline.predict_proba(trivial_test)
for cls, prob in zip(router_pipeline.classes_, probs[0]):
    print(f"  {cls}: {prob * 100:.2f}%")

  0: 60.04%
  1: 4.25%
  2: 35.71%
